# Silver Layer - Budget Table Cleaning

## Source & Destination
- **Source**: `automobile_repair.bronze.ns_budget`
- **Destination**: `automobile_repair.silver.budget`

## 7 Essential Columns for Revenue vs Budget KPI

### Identifiers (1)
1. `store_id` - Store ID (renamed from ns_store_id)

### Budget Data (2)
2. `month` - Budget month (DATE format)
3. `budget_amount` - Budget target amount

### Time Dimensions (2)
4. `budget_year` - Year for filtering
5. `budget_month` - Month number (1-12) for filtering

### Manager Context (2)
6. `manager_id` - Manager responsible (from store table)
7. `manager_name` - Manager name (from store table)

## What Was Removed
- ❌ `approved_by` - Not needed for Revenue vs Budget KPI

## Data Quality
✅ **Perfect Data** - No NULLs, duplicates, or invalid amounts
✅ **All 50 stores** have matching records in store table
✅ **25 months** of budget data (Jan 2024 - Jan 2026)

In [0]:
%sql
-- Silver Budget Transformation
-- For Revenue vs Budget KPI: Compare monthly revenue vs budget by manager

CREATE SCHEMA IF NOT EXISTS automobile_repair.silver;

CREATE OR REPLACE TABLE automobile_repair.silver.budget AS
SELECT 
    -- Identifier (1) - renamed for consistency
    b.ns_store_id AS store_id,
    
    -- Budget data (2)
    b.month,
    b.budget_amount,
    
    -- Time dimensions (2)
    YEAR(b.month) AS budget_year,
    MONTH(b.month) AS budget_month,
    
    -- Manager context (2) - from store table
    s.manager_id,
    s.manager_name
    
FROM automobile_repair.bronze.ns_budget b
LEFT JOIN automobile_repair.bronze.store s ON b.ns_store_id = s.store_id
ORDER BY b.month DESC, b.ns_store_id;

In [0]:
%sql
-- Verify the 7-column table structure
-- Show recent budget records with manager info
SELECT *
FROM automobile_repair.silver.budget
ORDER BY month DESC, store_id
LIMIT 20;

In [0]:
%sql
-- Monthly budget by manager
-- Ready to compare against actual revenue
SELECT 
    budget_year,
    budget_month,
    manager_name,
    COUNT(DISTINCT store_id) as stores_managed,
    SUM(budget_amount) as total_budget,
    ROUND(AVG(budget_amount), 2) as avg_budget_per_store
FROM automobile_repair.silver.budget
GROUP BY budget_year, budget_month, manager_name
ORDER BY budget_year DESC, budget_month DESC, total_budget DESC
LIMIT 20;